In [2]:
import pandas as pd
import re
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
archivo = 'Sedes_600.xlsx'
df = pd.read_excel(archivo)

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
pd.set_option('display.max_columns', None)

In [ ]:
df.info()

In [ ]:
# CUMPLE = 75 - 100
# NO CUMPLE = 50 - 74
# DEFICIENTE = 0 - 49

# PARA PRIORIZAR SE TOMA LAS DE PUNTO MAS BAJITO

In [ ]:
calificaciones = df.notna().replace({True: 100, False: 49})
promedio = calificaciones.mean(axis=1)
df['puntaje_diligenciamiento'] = promedio
        

In [ ]:
df

In [ ]:
# columnas numericas del datasets en lista para limpieza
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(num_cols)
len(num_cols)

In [ ]:
# Columnas categoricas del dataset en lista para limpieza
cat_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()
print(cat_cols)
len(cat_cols)

In [ ]:
# funcion para contar respuestas
def contar_respuestas(columna):
    
    df['Cuenta_' + columna]= df[columna].str.split(';').str.len()
    return df


In [ ]:
# contar respuestas de 'Jornada'
contar_respuestas('Jornada')

In [ ]:
# puntaje de jornada
df['puntaje_jornada'] = np.where(df['Cuenta_Jornada'] > 3, 100,
                        np.where(df['Cuenta_Jornada'] == 2, 75, 49))

In [ ]:
# Calificar 'Nivel Educativo'
contar_respuestas('Nivel Educativo')
df["Puntaje_nivel_educativo"] = np.where(df['Cuenta_Nivel Educativo'] > 3, 100,
                                np.where(df['Cuenta_Nivel Educativo'] == 2, 75, 49))

In [ ]:
# Puntaje Semillas Capacidades Diversas
df['Puntaje_Semillas_Capacidades_Diversas'] = np.where(df['Número de Semillas con Capacidades Diversas'] > 0, 100,
                                            np.where(df['Número de Semillas con Capacidades Diversas'] == 0, 49, 0))

In [ ]:
#Calificacion 'Semillas Casa Permanencia'
df['Puntaje_Semillas_Casa_Permanencia'] = np.where(df['Número de semillas en Casa de Permanencia'] > 0, 100,
                                        np.where(df['Número de semillas en Casa de Permanencia'] == 0, 49, 0))

In [ ]:
df['Puntaje_URL_Predio'] = np.where(df['URL Documento Predio'].str.contains('http', na=False), 100,49)

In [ ]:
Respuestas_estado_predio = {'La sede educativa no cuenta con documentos de legalización o tenencia del predio ya sea por vía de la ley ordinaria o derecho propio.':49,
                            'La sede educativa cuenta con un documento que acredita legalización del predio, sin embargo, presenta tramites o procesos administrativos y/o jurídicos pendientes para quedar plenamente legalizada.':74,
                            'La sede educativa cuenta con documentación legal vigente y completa que acredita la tenencia de la tierra y legalización del predio, conforme a la ley ordinaria o derecho propio.':100}

df['Puntaje_Estado_Predio'] = df['¿En qué estado se encuentra la legalización del predio de la sede educativa, según la ley ordinaria o el derecho propio?'].map(Respuestas_estado_predio).fillna(0)

In [ ]:
respuesta_predio_arrendado = {'Sí': 100, 'No':49}
df['Puntaje_Predio_Arrendado'] = df['Predio Arrendado'].map(respuesta_predio_arrendado).fillna(0)

In [ ]:
contar_respuestas('Medios de Transporte')


In [ ]:
df["Puntaje_Medios_Transporte"] = np.where(df['Cuenta_Medios de Transporte'] >= 3, 100,
                                np.where(df['Cuenta_Medios de Transporte'] == 1, 49, 0))

In [ ]:
# Diccionario de puntajes por condición
puntajes_condicion = {
    "Buenas condiciones": 100,
    "Condiciones regulares": 74,
    "Condiciones Críticas": 49
}

def calcular_puntaje_accesos_predio(texto):
    
    if pd.isna(texto):
        return 0
    puntajes = []

    # Buscar cada bloque entre []
    bloques = re.findall(r'\[(.*?)\]', texto)
    
    for bloque in bloques:

        # -----------------------------
        # CALIFICAR CONDICIÓN
        # -----------------------------
        if "Buenas condiciones" in bloque:
            puntajes.append(100)

        elif "Condiciones regulares" in bloque:
            puntajes.append(74)

        elif "Condiciones Críticas" in bloque:
            puntajes.append(49)
            
        # -----------------------------
        # CALIFICAR DISTANCIA
        # -----------------------------
        distancia_match = re.search(r'Distancia:\s*([\d\.]+)', bloque)

        if distancia_match:
            distancia = float(distancia_match.group(1))

            if distancia > 30:
                puntajes.append(100)
            else:
                puntajes.append(49)

    # Promediar todos los puntajes encontrados
    if puntajes:
        return round(sum(puntajes) / len(puntajes), 2)

    return None

accesos = str(df.loc[0, 'Accesos y Condiciones Predio'])
df['Puntaje_Accesos_Condiciones_Predio'] = df['Accesos y Condiciones Predio'].apply(calcular_puntaje_accesos_predio)


In [ ]:
# Diccionario de puntajes por condición
puntajes_condicion = {
    "Riesgo controlado con mitigación adecuada": 100,
    "Riesgo medio con mitigación parcial,": 74,
    "Riesgo alto sin mitigación": 49
}

def calcular_puntaje_acceso_antropicos(texto):
    
    if pd.isna(texto):
        return 0
    puntajes = []

    # Buscar cada bloque entre []
    bloques = re.findall(r'\[(.*?)\]', texto)
    
    for bloque in bloques:

        # -----------------------------
        # CALIFICAR CONDICIÓN
        # -----------------------------
        if "Riesgo controlado con mitigación adecuada" in bloque:
            puntajes.append(100)

        elif "Riesgo medio con mitigación parcial" in bloque:
            puntajes.append(74)

        elif "Riesgo alto sin mitigación" in bloque:
            puntajes.append(49)
            
            
    # Promediar todos los puntajes encontrados
    if puntajes:
        return round(sum(puntajes) / len(puntajes), 2)

    return None
df['Puntaje_Accesos y Condiciones Riesgos Naturales'] = df['Accesos y Condiciones Riesgo Natural'].apply(calcular_puntaje_acceso_antropicos)

In [ ]:
# Diccionario de puntajes por condición
puntajes_condicion = {
    "Ubicación de condición segura": 100,
    "Ubicación en condición limite": 74,
    "Ubicación en condición crítica": 49
}

def calcular_puntaje_acceso_antropicos(texto):
    
    if pd.isna(texto):
        return 0
    puntajes = []

    # Buscar cada bloque entre []
    bloques = re.findall(r'\[(.*?)\]', texto)
    
    for bloque in bloques:

        # -----------------------------
        # CALIFICAR CONDICIÓN
        # -----------------------------
        if "Ubicación de condición segura" in bloque:
            puntajes.append(100)

        elif "Ubicación en condición limite" in bloque:
            puntajes.append(74)

        elif "Ubicación en condición crítica" in bloque:
            puntajes.append(49)
        
        # -----------------------------
        # CALIFICAR DISTANCIA
        # -----------------------------
        distancia_match = re.search(r'Distancia:\s*([\d\.]+)', bloque)

        if distancia_match:
            distancia = float(distancia_match.group(1))

            if distancia > 100:
                puntajes.append(100)
            else:
                puntajes.append(49)
            
    # Promediar todos los puntajes encontrados
    if puntajes:
        return round(sum(puntajes) / len(puntajes), 2)

    return None
df['Puntaje_Accesos y Condiciones Riesgos Antrópicos'] = df['Accesos y Condiciones Riesgos Antrópicos'].apply(calcular_puntaje_acceso_antropicos)

In [ ]:
# Diccionario de puntajes por condición
puntajes_condicion = {
    "Condición adecuada": 100,
    "Condiciones parciales": 74,
    "Condiciones críticas": 49
}

def calcular_puntaje_acceso_cerramiento(texto):
    
    if pd.isna(texto):
        return 0
    puntajes = []

    # Buscar cada bloque entre []
    bloques = re.findall(r'\[(.*?)\]', texto)
    
    for bloque in bloques:

        # -----------------------------
        # CALIFICAR CONDICIÓN
        # -----------------------------
        if "Condición adecuada" in bloque:
            puntajes.append(100)

        elif "Condiciones parciales" in bloque:
            puntajes.append(74)

        elif "Condiciones críticas" in bloque:
            puntajes.append(49)
            
    # Promediar todos los puntajes encontrados
    if puntajes:
        return round(sum(puntajes) / len(puntajes), 2)

    return None
df['Puntaje_Accesos y Condiciones Cerramiento Perimetral'] = df['Accesos y Condiciones Cerramiento Perimetral'].apply(calcular_puntaje_acceso_cerramiento)

In [ ]:
respuestas_condiciones_servicio_agua = {'el servicio no es permanente. el suministro de agua es irregular o limitado y afecta significativamente el desarrollo de las actividades educativas y las condiciones de higiene.':49,
                                        'el servicio funciona parcialmente. el suministro de agua está disponible la mayor parte del tiempo, pero presenta interrupciones que afectan parcialmente las actividades de la sede.':74,
                                        'el servicio es permanente. el suministro de agua es continuo y garantiza las condiciones necesarias para el funcionamiento de la sede educativa y la atención de la comunidad':100
                                        }

df['Puntaje_Condiciones_Servicio_Agua'] = df['Condiciones Servicio Público'].map(respuestas_condiciones_servicio_agua).fillna(0)

In [ ]:
df

In [ ]:
def calcular_puntaje_componentes_agua(texto):
    
    # si es NaN devolver 0
    if pd.isna(texto):
        return 0
    
    puntajes = []
    
    texto = str(texto)
    
    # Calificar antiguedad
    
    if "1 a 5 años" in texto:
        puntajes.append(100)
    
    elif "6 a 10 años" in texto:
        puntajes.append(74)
        
    elif "11 a 20 años" in texto:
        puntajes.append(74)
        
    elif "Mas de 20 años" in texto:
        puntajes.append(49)
        
    # Calificar estado
    
    if "Buen Estado" in texto:
        puntajes.append(100)
    
    elif "Funciona, Requiere Adecuaciones" in texto:
        puntajes.append(74)
        
    elif "Alto Deterioro" in texto:
        puntajes.append(49)
        
    if puntajes:
        return round(sum(puntajes) / len(puntajes), 2)
    
    return 0

df["Puntaje_componentes_presentes_suministro_agua"] = df["Componentes presentes en suministro agua"].apply(calcular_puntaje_componentes_agua)

In [ ]:
def calcular_puntaje_componentes_alcantarillado(texto):
    
    if pd.isna(texto):
        return 0
    
    puntajes = []
    
    texto = str(texto)
    
    # Calificar Antiguedad
    
    if "1 a 5 años" in texto:
        puntajes.append(100)
        
    elif "6 a 10 Años" in texto:
        puntajes.append(74)
        
    elif "11 a 20 Años" in texto:
        puntajes.append(74)
        
    elif "Mas de 20 Años" in texto:
        puntajes.append(49)
        
    # Calificar Estado 
    
    if "Buen Estado" in texto:
        puntajes.append(100)
        
    elif "Funciona, Requiere Adecuaciones" in texto:
        puntajes.append(74)
        
    elif "Alto Deterioro" in texto:
        puntajes.append(49)
        
    # Promedio Final 
    
    if puntajes:
        
        return round(sum(puntajes) / len(puntajes), 2)
    
    return 0

df["Puntaje_componentes_presentes_alcantarillado"] = df["Componentes Presentes Alcantarillado"].apply(calcular_puntaje_componentes_alcantarillado)

In [ ]:
def caluclar_puntaje_aparatos_sanitarios(texto):
    
    # Si es NaN 
    
    if pd.isna(texto):
        return pd.Series([0, "", ""])
    
    puntajes = []
    tipos = []
    cantidades = []
    
    # Separar respuestas por ;
    bloques = str(texto).split(";")
    
    for bloque in bloques:
        
        # Extraer tipo
        
        tipo_match = re.search(r'Tipo:\s*([^,]+)', bloque)
        
        if tipo_match:
            tipos.append(tipo_match.group(1).strip())
            
        # Extraer Cantidad
        
        cantidad_match = re.search(r'Cantidad:\s*(\d+)', bloque)
        
        if cantidad_match:
            cantidades.append(cantidad_match.group(1))
            
        # Calificar estado
        
        if "Funciona adecuadamente" in bloque:
            puntajes.append(100)
            
        elif "Funcionamiento intermitente" in bloque:
            puntajes.append(74)
            
        elif "No funciona" in bloque:
            puntajes.append(49)
            
        promedio = round(sum(puntajes) / len(puntajes), 2) if puntajes else 0
        
        return pd.Series([
            promedio,
            ", ".join(tipos),
            ", ".join(cantidades)
        ])
        
        # Crear columnas 
        
df[["Puntaje_tipo_aparato_hidrosanitario",
    "Tipo_aparato_sanitario", 
            "Cantidad_aparato_sanitario",
            ]] = df["Tipo Aparato Hidrosanitario"].apply(caluclar_puntaje_aparatos_sanitarios)

In [ ]:
def calcular_puntaje_aguas_lluvias(texto):
    
    if pd.isna(texto):
        return 0
    
    puntajes = []
    
    texto = str(texto)
    
    if "Sí" in texto:
        puntajes.append(100)
        
    elif "No" in texto:
        puntajes.append(49)
        
    if puntajes:
        return round(sum(puntajes) / len(puntajes), 0)
    return 0

df["Puntaje_aguas_lluvias"] = df["Aguas Lluvias"].apply(calcular_puntaje_aguas_lluvias)

In [ ]:
respuestas_suministro_gas = {"Red Pública":100, "Tanque fijo":100, "No tiene":49, "Otro":49, "Cilindro":74, "Biodigestor":74}
df["Puntaje_suministro_gas"] = df["Fuente Suministro Gas"].map(respuestas_suministro_gas)

In [ ]:
def calcular_puntaje_suministro_energia(texto):
    
    if pd.isna(texto):
        return 0
    
    puntajes = []
    
    bloques = re.findall(r'\[(.*?)\]', texto)
    
    for bloque in bloques:
    
    #Calificar el tipo de transformador 
    
        if "Propio" in bloque:
            puntajes.append(100)
            
        elif "Comunitario" in bloque:
            puntajes.append(74)
            
        elif "No aplica" in bloque:
            puntajes.append(49)
            
        # Calificar el funcionamiento de servicio
        
        if "Servicio menos del 50%" in bloque:
            puntajes.append(49)
            
        elif "Servicio al 50%" in bloque:
            puntajes.append(74)
            
        elif "Servicio al 100%" in bloque:
            puntajes.append(100)
            
        
    if puntajes:
        return round(sum(puntajes) / len(puntajes), 0)
    
    else:
        return 0
    
df["Puntaje_suministro_energia"] = df["Suministro de Energía"].apply(calcular_puntaje_suministro_energia)

In [ ]:
respuestas_estado_energia_solar = {"El sistema funciona adecuadamente, suministra energía de forma continua y no presenta fallas visibles ni interrupciones.":100,
                                "El sistema presenta fallas ocasionales, disminución en el rendimiento o requiere mantenimiento preventivo para garantizar su funcionamiento óptimo.":74,
                                "El sistema no funciona correctamente, presenta daños evidentes o no está suministrando energía, requiriendo reparación o reemplazo.":49}
df["Puntaje_estado_energia_solar"] = df["¿Cuál es el estado del sistema de energía solar instalado en la sede educativa?"].map(respuestas_estado_energia_solar).fillna(0)

In [ ]:
respuestas_estado_planta_electrica = {"La planta eléctrica funciona correctamente, enciende sin dificultad, suministra energía de manera estable y no presenta ruidos anormales, fugas ni fallas visibles. Cuenta con mantenimiento al día.":100,
                                    "La planta eléctrica funciona, pero presenta algunas fallas menores como dificultad en el arranque, funcionamiento inestable, ruidos inusuales o mantenimiento pendiente. Requiere revisión o mantenimiento correctivo.":74,
                                    "La planta eléctrica no funciona, presenta fallas graves o daños visibles en componentes principales (motor, tablero, sistema eléctrico o estructura). Requiere reparación mayor o reemplazo.":49}
df["Puntaje_estado_planta_electrica"] = df["¿Cuál es el estado de funcionamiento de la Planta Eléctrica de la sede educativa?"].map(respuestas_estado_planta_electrica).fillna(0)

In [ ]:
respuestas_acometida_principal = {"Convencional":100, "No convencional":49}
df["Puntaje_acometida_principal"] = df["Tipo Acometida Redes Internas"].map(respuestas_acometida_principal).fillna(0)

In [ ]:
respuestas_estado_acometida_electrica = {"La acometida se encuentra en buen estado, con cableado adecuado, conexiones seguras, sin empalmes improvisados ni cables expuestos, y cumple condiciones básicas de seguridad.":100,
                                        "La acometida presenta desgaste, empalmes visibles o condiciones que requieren mantenimiento preventivo o correctivo, pero aún se encuentra operativa.":74,
                                        "La acometida presenta daños evidentes, cables expuestos, conexiones inseguras o instalaciones improvisadas que representan riesgo eléctrico y requieren intervención inmediata.":49}
df["Puntaje_estado_acometida_electrica"] = df["¿Cuál es el estado de acometida principal de energía eléctrica?"].map(respuestas_estado_acometida_electrica).fillna(0)

In [ ]:
respuestas_estado_iluminacion = {"Funciona adecuadamente. Los puntos de iluminación se encuentran en buen estado, operan correctamente y no presentan riesgos visibles como cables expuestos, sobrecalentamiento o fallas eléctricas.":100,
                                "Funcionamiento intermitente. Algunos puntos de iluminación presentan fallas ocasionales, bajo rendimiento o requieren mantenimiento.":74,
                                "No funciona o presenta deterioro. Varios puntos de iluminación no funcionan o presentan daños visibles que pueden representar riesgo eléctrico y requieren reparación o reemplazo.":49}
df["Puntaje_estado_iluminacion"] = df["Estado Iluminación"].map(respuestas_estado_iluminacion).fillna(0)

In [ ]:
respuestas_estado_tomacorrientes = {"Funcionan adecuadamente. Los tomacorrientes se encuentran en buen estado, operan correctamente y no presentan riesgos visibles.":100,
                                    "Funcionamiento intermitente. Algunos tomacorrientes presentan fallas ocasionales, conexiones inestables o requieren mantenimiento.":74,
                                    "No funcionan o presentan deterioro. Varios tomacorrientes no funcionan o presentan daños visibles que pueden representar riesgo eléctrico y requieren reparación o reemplazo.":49}
df["Puntaje_estado_tomacorrientes"] = df["Estado Tomacorrientes"].map(respuestas_estado_tomacorrientes).fillna(0)

In [ ]:
respuestas_estado_tablero = {"El tablero se encuentra en buen estado, correctamente cerrado y protegido, con breakers identificados, cableado organizado, sin sobrecalentamiento ni riesgos visibles.":100,
                            "l tablero funciona, pero presenta desorden en el cableado, falta de identificación en los circuitos, tapas deterioradas o requiere mantenimiento preventivo.":74,
                            "El tablero presenta daños estructurales, ausencia de tapa o protección, cableado expuesto, conexiones improvisadas, breakers defectuosos o condiciones que representan riesgo eléctrico.":49}

df["Puntaje_estado_tablero"] = df["¿Cuál es el estado del tablero de distribución eléctrica?"].map(respuestas_estado_tablero).fillna(0)

In [ ]:
respuestas_estado_circuitos = {"Los circuitos eléctricos funcionan de manera estable y continua, sin interrupciones frecuentes, sin sobrecargas evidentes y garantizan el suministro adecuado en todos los espacios.":100,
                            "Se presentan fallas ocasionales, fluctuaciones de voltaje o interrupciones parciales en algunos espacios. Requiere revisión o mantenimiento preventivo.":74,
                            "Existen fallas frecuentes, circuitos fuera de servicio, sobrecargas, disparo constante de breakers o riesgos eléctricos que afectan el funcionamiento normal de la sede.":49
                            }

df["Puntaje_estado_circuitos"] = df["¿Funcionamiento general de los circuitos eléctricos en la sede educativa?"].map(respuestas_estado_circuitos).fillna(0)

In [ ]:
respuestas_material_tuberia_electrica = {"PVC":74, "Metálico":74, "Sin canalización":49}
df["Puntaje_material_tuberia_electrica"] = df["Tipo Material Tubería Eléctrica"].map(respuestas_material_tuberia_electrica).fillna(0)

In [ ]:
respuestas_tipo_comunicacion = {"Wifi":74, "Punto a punto":74, "Satelital":100, "Red publica":100, "No tiene":49}
df["Puntaje_tipo_comunicacion"] = df["Tipo Comunicaciónes"].map(respuestas_tipo_comunicacion).fillna(0)

In [ ]:
respuestas_estado_comunicaciones = {"El servicio de voz y datos funciona de manera estable y continua, con buena cobertura en la sede, velocidad adecuada y sin interrupciones frecuentes.":100,
                                "El servicio presenta intermitencias ocasionales, baja velocidad, problemas de cobertura en algunos espacios o fallas esporádicas en la comunicación.": 74,
                                "El servicio es inestable o inexistente, presenta interrupciones constantes, mala calidad en la comunicación o no cubre los espacios necesarios para el funcionamiento educativo.":49
                                }

df["Puntaje_estado_comunicaciones"] = df["¿Como es el funcionamiento de telecomunicaciones (voz y datos) de la sede educativa?"].map(respuestas_estado_comunicaciones).fillna(0)

In [ ]:
respuestas_huerta_escolar = {"Sí":100, "No":49}
df["Puntaje_huerta_escolar"] = df["¿La sede educativa tiene huerta escolar?"].map(respuestas_huerta_escolar).fillna(0)


In [ ]:
respuestas_granja_escolar = {"No cuenta con granja escolar o esta no se encuentra en funcionamiento.":49,
                            "Cuenta con granja escolar, pero su funcionamiento es parcial, intermitente o no está completamente articulada al proceso pedagógico.":74,
                            "Cuenta con granja escolar activa, en funcionamiento permanente, articulada al proceso pedagógico y con participación de las semillas (estudiantes).":100}
df["Puntaje_granja_escolar"] = df["¿La sede educativa tiene granja escolar?"].map(respuestas_granja_escolar).fillna(0)

In [ ]:
respuestas_procesos_madre_tierra = {"La sede no desarrolla acciones claras de cuidado y preservación ambiental o estas son inexistentes.":49,
                                "La sede realiza algunas actividades ambientales de manera ocasional, pero no cuenta con un proceso estructurado o permanente.":74,
                                "La sede desarrolla procesos permanentes y organizados de cuidado ambiental (huertas, reforestación, manejo de residuos, protección de fuentes hídricas, proyectos pedagógicos ambientales, prácticas culturales propias de armonización con la Madre Tierra).":100}

df["Puntaje_procesos_madre_tierra"] = df["¿La sede educativa desarrolla procesos de cuidado y preservación de los ecosistemas de la madre tierra y fauna?"].map(respuestas_procesos_madre_tierra).fillna(0)

In [ ]:
def puntaje_residuos_inorganicos(texto):
    
    if pd.isna(texto):
        return 0
    
    puntajes = []
    
    texto = str(texto)
    
    if "Sí" in texto:
        puntajes.append(100)
        
    elif "No" in texto:
        puntajes.append(49)
        
        
    if puntajes:
        return round(sum(puntajes) / len(puntajes), 0)
    
    else:
        return 0
    
df["Puntaje_residuos_inorganicos"] =df["Separación de Residuos Inorgánicos"].apply(puntaje_residuos_inorganicos)
        
    

In [ ]:
def puntaje_residuos_organicos(texto):
    
    if pd.isna(texto):
        return 0
    
    puntajes = []
    
    texto = str(texto)
    
    if "Sí" in texto:
        puntajes.append(100)
        
    elif "No" in texto:
        puntajes.append(49)
        
        
    if puntajes:
        return round(sum(puntajes) / len(puntajes), 0)
    
    else:
        return 0
    
df["Puntaje_residuos_organicos"] =df["Separación de Residuos Orgánicos"].apply(puntaje_residuos_organicos)

In [ ]:
puntajes_arquitectura_propia = {"Sí":100, "No":49}

df["Puntaje_arquitectura_propia"] = df["Arquitectura Propia"].map(puntajes_arquitectura_propia).fillna(0)

In [ ]:
def contar_respuestas_1(columna):
    
    df["Cuenta_" + columna] = df[columna].str.split(",").str.len()
    return df

In [ ]:
contar_respuestas_1("Tipo de Arquitectura")

df["Puntaje_tipo_arquitectura"] = np.where(df["Cuenta_Tipo de Arquitectura"] > 3, 100,
                                np.where(df["Cuenta_Tipo de Arquitectura"] == 2, 75, 49))

In [ ]:
respuesta_acceso_semillas_diversas = {"Sí":100, "No": 49}

df["Puntaje_acceso_semillas_diversas"] = df["Acceso a Semillas con Capacidades Diversas"].map(respuesta_acceso_semillas_diversas).fillna(0)

In [ ]:
contar_respuestas_1("Elementos Complementarios de Acceso")


df["Puntaje_elementos_complementarios"] = np.where(df["Cuenta_Elementos Complementarios de Acceso"] > 2, 100,
    np.where(df["Cuenta_Elementos Complementarios de Acceso"] == 2, 75, 49))


In [ ]:
respuestas_mantenimiento_infraestructura = {"Se realiza mantenimiento preventivo y correctivo de manera periódica, planificada y con responsables definidos.":100,
                                            "Se realiza mantenimiento ocasional o solo cuando se presentan daños (correctivo no planificado).":74,
                                            "No se realiza mantenimiento o la infraestructura se encuentra desatendida.":49}
df["Puntaje_mantenimiento_infraestructura"] = df["¿Como se realiza el mantenimiento de la infraestructura y de las instalaciones escolares en la sede educativa?"].map(respuestas_mantenimiento_infraestructura).fillna(0)

In [ ]:
respuestas_condicion_fisica_ruta = {"Las rutas se encuentran en buen estado, libres de obstáculos y permiten una evacuación segura.":100,
                                "Presentan deterioro parcial, requieren mantenimiento o tienen algunas condiciones que pueden generar riesgo.":74,
                                "Presentan daños significativos, condiciones inseguras o no existen rutas de evacuación.":49}

df["Puntaje_condicion_fisica_ruta"] = df["Condición Física de Ruta de Evacuación"].map(respuestas_condicion_fisica_ruta).fillna(0)

In [ ]:
respuesta_iluminacion_exterior = {"El sistema cubre la totalidad del exterior y funciona correctamente.":100,
                                "El sistema está instalado parcialmente o requiere mantenimiento para su funcionamiento adecuado.":74,
                                "El sistema no funciona o se encuentra fuera de servicio.":49}

df["Puntaje_iluminaicion_exterior"] = df["Sistema iluminación Exterior"].map(respuesta_iluminacion_exterior).fillna(0)

In [ ]:
respuestas_planimetria_sede = {"Sí":100, "No": 49}

df["Puntaje_planimetria_sede"] = df["Planimetría Sede"].map(respuestas_planimetria_sede).fillna(0)

In [ ]:
respuestas_plan_evacuacion = {"Sí, existe y está socializado con la comunidad educativa.":100, 
                            "Existe, pero no está socializado o actualizado":74,
                            "No existe plan de evacuación.":49}

df["Puntaje_plan_evacuacion"] = df["¿La sede educativa cuenta con un plan de evacuación definido y conocido por la comunidad educativa?"].map(respuestas_plan_evacuacion).fillna(0)

In [ ]:
respuestas_rutas_evacuacion = {"Todas las rutas están señalizadas correctamente.":100,
                            "Algunas rutas están señalizadas.":74,
                            "No existen señales de evacuación": 49}

df["Puntaje_rutas_evacuacion"] = df["¿Las rutas de evacuación dentro de la sede educativa están señalizadas y visibles?"].map(respuestas_rutas_evacuacion).fillna(0)

In [ ]:
respuestas_simulacros_evacuacion = {"Sí, se realizan periódicamente.":100,
                                    "Se han realizado ocasionalmente.":74,
                                    "No se realizan simulacros.":49}

df["Puntaje_simulacros_evacuacion"] = df["¿La sede educativa realiza simulacros de evacuación o actividades de preparación ante emergencias?"].map(respuestas_simulacros_evacuacion).fillna(0)

In [ ]:
respuestas_punto_encuentro = {"Sí, está definido y señalizado.":100,
                            "Está definido, pero no señalizado.":74,
                            "No existe punto de encuentro.":49}

df["Puntaje_punto_encuentro"] = df["¿La sede educativa tiene definido un punto de encuentro o zona segura para emergencias?"].map(respuestas_punto_encuentro).fillna(0)

In [ ]:
def calificar_cantidad(cantidad):
    """
    Regla:
    > 3 = 49
    = 2 o 3 = 74
    = 1 = 100
    """
    if cantidad > 3:
        return 49
    elif cantidad >= 2:
        return 74
    elif cantidad == 1:
        return 100
    return 0


def procesar_espacios(texto):

    # Si es NaN
    if pd.isna(texto):
        return pd.Series([
            0, 0, 0, 0, 0,
            "", 0
        ])

    # ----------------------------------------
    # EXTRAER BLOQUES []
    # ----------------------------------------
    bloques = re.findall(r'\[(.*?)\]', str(texto))

    puntaje_ocupacion = []
    puntaje_muros = []
    puntaje_fachada = []
    puntaje_antiguedad = []
    puntaje_estado = []

    areas = []
    suma_areas = 0

    for bloque in bloques:

        # ========================================
        # OCUPACIÓN O USO
        # ========================================
        ocupacion_match = re.search(
            r'Ocupación o uso:\s*(.*?);',
            bloque
        )

        if ocupacion_match:

            ocupaciones = [
                x.strip()
                for x in ocupacion_match.group(1).split(",")
                if x.strip()
            ]

            cantidad = len(ocupaciones)

            puntaje_ocupacion.append(
                calificar_cantidad(cantidad)
            )

        # ========================================
        # MUROS
        # ========================================
        muros_match = re.search(
            r'Muros:\s*(.*?);',
            bloque
        )

        if muros_match:

            muros = [
                x.strip()
                for x in muros_match.group(1).split(",")
                if x.strip()
            ]

            cantidad = len(muros)

            puntaje_muros.append(
                calificar_cantidad(cantidad)
            )

        # ========================================
        # FACHADA
        # ========================================
        fachada_match = re.search(
            r'Fachada:\s*(.*?);',
            bloque
        )

        if fachada_match:

            fachadas = [
                x.strip()
                for x in fachada_match.group(1).split(",")
                if x.strip()
            ]

            cantidad = len(fachadas)

            puntaje_fachada.append(
                calificar_cantidad(cantidad)
            )

        # ========================================
        # ANTIGÜEDAD
        # ========================================
        if "1 a 5 Años" in bloque:
            puntaje_antiguedad.append(100)

        elif "6 a 10 Años" in bloque:
            puntaje_antiguedad.append(74)

        elif "11 a 20 Años" in bloque:
            puntaje_antiguedad.append(60)

        elif "Mas de 20 años" in bloque:
            puntaje_antiguedad.append(49)

        # ========================================
        # ESTADO ACTUAL
        # ========================================
        if "Bueno: sin afectaciones visuales" in bloque:
            puntaje_estado.append(100)

        elif "Regular: Presenta grietas superficiales" in bloque:
            puntaje_estado.append(74)

        elif "Malo: Riesgo evidente de habitabilidad" in bloque:
            puntaje_estado.append(49)

        # ========================================
        # ÁREA
        # ========================================
        area_match = re.search(
            r'Área Total:\s*([\d\.]+)',
            bloque
        )

        if area_match:

            area = float(area_match.group(1))

            areas.append(area)

            suma_areas += area

    # ========================================
    # PROMEDIOS
    # ========================================
    def promedio(lista):
        return round(sum(lista) / len(lista), 2) if lista else 0

    return pd.Series([

        promedio(puntaje_ocupacion),
        promedio(puntaje_muros),
        promedio(puntaje_fachada),
        promedio(puntaje_antiguedad),
        promedio(puntaje_estado),

        ", ".join(map(str, areas)),
        suma_areas
    ])


# ----------------------------------------
# CREAR COLUMNAS
# ----------------------------------------

df[
    [
        "Puntaje_ocupacion_uso",
        "Puntaje_muros",
        "Puntaje_fachada",
        "Puntaje_antiguedad",
        "Puntaje_estado_actual",
        "Areas_por_espacio",
        "Area_total_espacios"
    ]
] = df["Espacios Cerrados"].apply(procesar_espacios)

In [ ]:

# ----------------------------------------
# FUNCIÓN PARA CALIFICAR CANTIDADES
# ----------------------------------------

def calificar_cantidad(cantidad):

    if cantidad > 3:
        return 49

    elif cantidad == 2:
        return 74

    elif cantidad == 1:
        return 100

    return 0


# ----------------------------------------
# FUNCIÓN PRINCIPAL
# ----------------------------------------

def procesar_espacios_abiertos(texto):

    # Si es NaN
    if pd.isna(texto):
        return pd.Series([
            0, 0, 0,
            "", 0
        ])

    # Extraer bloques []
    bloques = re.findall(r'\[(.*?)\]', str(texto))

    puntajes_uso = []
    puntajes_antiguedad = []
    puntajes_estado = []

    areas = []
    suma_areas = 0

    for bloque in bloques:

        # ========================================
        # USO ESPACIO ABIERTO
        # ========================================
        uso_match = re.search(
            r'Uso Espacio Abierto:\s*(.*?);',
            bloque
        )

        if uso_match:

            usos = [
                x.strip()
                for x in uso_match.group(1).split(",")
                if x.strip()
            ]

            cantidad_usos = len(usos)

            puntajes_uso.append(
                calificar_cantidad(cantidad_usos)
            )

        # ========================================
        # ANTIGÜEDAD
        # ========================================
        bloque_lower = bloque.lower()

        if "1 a 5 años" in bloque_lower:
            puntajes_antiguedad.append(100)

        elif "6 a 10 años" in bloque_lower:
            puntajes_antiguedad.append(74)

        elif "11 a 20 años" in bloque_lower:
            puntajes_antiguedad.append(60)

        elif "mas de 20 años" in bloque_lower:
            puntajes_antiguedad.append(49)

        # ========================================
        # ESTADO ACTUAL
        # ========================================
        if "Bueno: Sin afectaciones visibles" in bloque:
            puntajes_estado.append(100)

        elif "Regular: Presenta desgaste" in bloque:
            puntajes_estado.append(74)

        elif "Malo: Presenta daños significativos" in bloque:
            puntajes_estado.append(49)

        # ========================================
        # EXTRAER ÁREA
        # ========================================
        area_match = re.search(
            r'Área Total:\s*([\d\.]+)',
            bloque
        )

        if area_match:

            area = float(area_match.group(1))

            areas.append(area)

            suma_areas += area

    # ========================================
    # FUNCIÓN PROMEDIO
    # ========================================
    def promedio(lista):
        return round(sum(lista) / len(lista), 2) if lista else 0

    return pd.Series([

        promedio(puntajes_uso),
        promedio(puntajes_antiguedad),
        promedio(puntajes_estado),

        ", ".join(map(str, areas)),
        suma_areas
    ])


# ----------------------------------------
# CREAR COLUMNAS
# ----------------------------------------

df[
    [
        "Puntaje_uso_espacio_abierto",
        "Puntaje_antiguedad_espacio_abierto",
        "Puntaje_estado_actual_espacio_abierto",
        "Areas_espacios_abiertos",
        "Area_total_espacios_abiertos"
    ]
] = df["Espacios Abiertos"].apply(procesar_espacios_abiertos)

In [ ]:
# =========================================================
# FUNCIONES AUXILIARES
# =========================================================

def puntaje_cantidad(cantidad):

    if cantidad > 3:
        return 49

    elif cantidad == 2:
        return 74

    else:
        return 100


def extraer_respuesta(texto, pregunta):

    patron = rf'Pregunta\s+{re.escape(pregunta)}.*?\((.*?)\)'

    match = re.search(
        patron,
        texto,
        re.IGNORECASE
    )

    if match:

        respuesta = match.group(1).strip()

        # Si es NULL
        if respuesta.upper() == "NULL":
            return None

        return respuesta

    return None


def extraer_si_no(texto, pregunta):

    respuesta = extraer_respuesta(
        texto,
        pregunta
    )

    if respuesta:
        return respuesta.strip()

    return None


# =========================================================
# FUNCIÓN PRINCIPAL
# =========================================================

def procesar_respuestas(texto):

    resultados = {}

    # =====================================================
    # SI ES NaN
    # =====================================================

    if pd.isna(texto):

        columnas = [
            "Puntaje_p1",
            "Puntaje_p2",
            "Puntaje_p3",
            "Puntaje_p3_1",
            "Puntaje_p4",
            "Puntaje_p5",
            "Puntaje_p6",
            "Puntaje_p6_1",
            "Puntaje_p7",
            "Puntaje_p8",
            "Puntaje_p8_1",
            "Puntaje_p9",
            "Puntaje_p10",
            "Puntaje_p11",
            "Puntaje_p12",
            "Puntaje_p12_1",
            "Promedio_p12_2",
            "Promedio_p12_3",
            "Puntaje_p12_4",
            "Puntaje_p13",
            "Puntaje_p14",
            "Puntaje_p15",
            "Puntaje_p16"
        ]

        for col in columnas:
            resultados[col] = 0

        return resultados

    texto = str(texto)

    # =====================================================
    # PREGUNTA 1
    # =====================================================

    p1 = extraer_respuesta(texto, "1.")

    if p1:

        porcentajes = re.findall(
            r'([A-Za-zÁÉÍÓÚáéíóúñÑ]+)\s*,\s*(\d+)%',
            p1
        )

        if porcentajes:

            datos = {
                nombre: int(valor)
                for nombre, valor in porcentajes
            }

            indigena = datos.get(
                "Indígena",
                0
            )

            mayor = max(datos.values())

            resultados["Puntaje_p1"] = (
                100 if indigena == mayor else 49
            )

        else:
            resultados["Puntaje_p1"] = 0

    else:
        resultados["Puntaje_p1"] = 0

    # =====================================================
    # PREGUNTAS CON CONTEO
    # =====================================================

    preguntas_conteo = {
        "2.": "Puntaje_p2",
        "3.1": "Puntaje_p3_1",
        "4.": "Puntaje_p4",
        "5.": "Puntaje_p5",
        "6.1": "Puntaje_p6_1",
        "7.": "Puntaje_p7",
        "8.1": "Puntaje_p8_1",
        "9.": "Puntaje_p9",
        "11.": "Puntaje_p11",
        "12.1": "Puntaje_p12_1"
    }

    for pregunta, columna in preguntas_conteo.items():

        respuesta = extraer_respuesta(
            texto,
            pregunta
        )

        if respuesta:

            cantidad = len([
                x.strip()
                for x in respuesta.split(",")
                if x.strip()
            ])

            resultados[columna] = (
                puntaje_cantidad(cantidad)
            )

        else:
            resultados[columna] = 0

    # =====================================================
    # PREGUNTAS SI / NO
    # =====================================================

    preguntas_si_no = {
        "3.": "Puntaje_p3",
        "6.": "Puntaje_p6",
        "8.": "Puntaje_p8",
        "12.": "Puntaje_p12",
        "13.": "Puntaje_p13"
    }

    for pregunta, columna in preguntas_si_no.items():

        respuesta = extraer_si_no(
            texto,
            pregunta
        )

        if pregunta == "13.":

            if respuesta == "Sí":
                resultados[columna] = 49

            elif respuesta == "No":
                resultados[columna] = 100

            else:
                resultados[columna] = 0

        else:

            if respuesta == "Sí":
                resultados[columna] = 49

            elif respuesta == "No":
                resultados[columna] = 100

            else:
                resultados[columna] = 0

    # =====================================================
    # PREGUNTA 10
    # =====================================================

    p10 = extraer_respuesta(
        texto,
        "10."
    )

    if p10:

        porcentaje = re.search(
            r'(\d+)%',
            p10
        )

        if porcentaje:

            valor = int(
                porcentaje.group(1)
            )

            if valor > 80:
                resultados["Puntaje_p10"] = 49

            elif valor >= 50:
                resultados["Puntaje_p10"] = 74

            else:
                resultados["Puntaje_p10"] = 100

        else:
            resultados["Puntaje_p10"] = 0

    else:
        resultados["Puntaje_p10"] = 0

    # =====================================================
    # PREGUNTA 12.2
    # =====================================================

    p122 = re.search(
        r'Pregunta\s+12\.2.*?Hablan:\s*(\d+).*?Escriben:\s*(\d+)',
        texto,
        re.IGNORECASE
    )

    if p122:

        hablan = int(
            p122.group(1)
        )

        escriben = int(
            p122.group(2)
        )

        resultados["Promedio_p12_2"] = round(
            (hablan + escriben) / 2,
            2
        )

    else:
        resultados["Promedio_p12_2"] = 0

    # =====================================================
    # PREGUNTA 12.3
    # =====================================================

    p123 = re.search(
        r'Pregunta\s+12\.3.*?Hablan:\s*(\d+).*?Escriben:\s*(\d+)',
        texto,
        re.IGNORECASE
    )

    if p123:

        hablan = int(
            p123.group(1)
        )

        escriben = int(
            p123.group(2)
        )

        resultados["Promedio_p12_3"] = round(
            (hablan + escriben) / 2,
            2
        )

    else:
        resultados["Promedio_p12_3"] = 0

    # =====================================================
    # PREGUNTA 12.4
    # =====================================================

    p124 = extraer_respuesta(
        texto,
        "12.4"
    )

    if p124:

        numero = re.search(
            r'(\d+)',
            p124
        )

        if numero:

            valor = int(
                numero.group(1)
            )

            if valor >= 5:
                resultados["Puntaje_p12_4"] = 49

            elif valor >= 3:
                resultados["Puntaje_p12_4"] = 74

            else:
                resultados["Puntaje_p12_4"] = 100

        else:
            resultados["Puntaje_p12_4"] = 0

    else:
        resultados["Puntaje_p12_4"] = 0

    # =====================================================
    # PREGUNTA 14
    # =====================================================

    p14 = extraer_respuesta(
        texto,
        "14."
    )

    if p14:

        if "Bueno" in p14:
            resultados["Puntaje_p14"] = 100

        elif "Regular" in p14:
            resultados["Puntaje_p14"] = 74

        elif "Malo" in p14:
            resultados["Puntaje_p14"] = 49

        else:
            resultados["Puntaje_p14"] = 0

    else:
        resultados["Puntaje_p14"] = 0

    # =====================================================
    # PREGUNTA 15
    # =====================================================

    p15 = extraer_si_no(
        texto,
        "15."
    )

    if p15 == "Sí":
        resultados["Puntaje_p15"] = 100

    elif p15 == "No":
        resultados["Puntaje_p15"] = 49

    else:
        resultados["Puntaje_p15"] = 0

    # =====================================================
    # PREGUNTA 16
    # =====================================================

    p16 = extraer_respuesta(
        texto,
        "16."
    )

    if p16:

        if "Bueno" in p16:
            resultados["Puntaje_p16"] = 100

        elif "Regular" in p16:
            resultados["Puntaje_p16"] = 74

        elif "Malo" in p16:
            resultados["Puntaje_p16"] = 49

        else:
            resultados["Puntaje_p16"] = 0

    else:
        resultados["Puntaje_p16"] = 0

    return resultados


# =========================================================
# APLICAR AL DATAFRAME
# =========================================================

df_resultados = df["Pregunta - Respuesta"].apply(
    procesar_respuestas
)

# Convertir correctamente
df_resultados = pd.DataFrame(
    df_resultados.tolist()
)

# Unir
df = pd.concat(
    [df, df_resultados],
    axis=1
)

# Reemplazar NaN por 0
df = df.fillna(0)



In [ ]:
# =========================================================
# COLUMNAS DE PUNTAJES
# =========================================================

columnas_puntajes = [
    "Puntaje_p1",
    "Puntaje_p2",
    "Puntaje_p3",
    "Puntaje_p3_1",
    "Puntaje_p4",
    "Puntaje_p5",
    "Puntaje_p6",
    "Puntaje_p6_1",
    "Puntaje_p7",
    "Puntaje_p8",
    "Puntaje_p8_1",
    "Puntaje_p9",
    "Puntaje_p10",
    "Puntaje_p11",
    "Puntaje_p12",
    "Puntaje_p12_1",
    "Puntaje_p12_4",
    "Puntaje_p13",
    "Puntaje_p14",
    "Puntaje_p15",
    "Puntaje_p16"
]

# =========================================================
# CREAR PROMEDIO GENERAL
# =========================================================

df["Puntaje_SIPEP"] = (
    df[columnas_puntajes]
    .mean(axis=1)
    .round(2)
)



In [ ]:
df

In [ ]:
# Criterio de Hacinamiento

# Mayor a 1.80 NO Hacinamiento
# Menor a 1.65 Hacinamiento
# Entre 1.65 y 1.80 en condicion limite 

df["Criterio_hacinamiento"] = np.where(
    (df["Area_total_espacios"].isna()) |
    (df["Número de Semillas"].isna()) |
    (df["Número de Semillas"] == 0),

    0,

    df["Area_total_espacios"] / df["Número de Semillas"]
)

In [ ]:
# Puntaje de Hacinamiento

df["Puntaje_hacinamiento"] = np.where(
    df["Criterio_hacinamiento"].isna(),
    0,
    np.where(
        df["Criterio_hacinamiento"] > 1.80,
        100,
        np.where(
            df["Criterio_hacinamiento"] < 1.65,
            49,
            74
        )
    )
)

In [ ]:
# Cumplimiento

df['Puntaje_CULTURAL_PEDAGOGICO'] = (df['puntaje_jornada'].fillna(0) * 0.166 +
                                    df['Puntaje_nivel_educativo'].fillna(0) * 0.166 + 
                                    df['Puntaje_Semillas_Capacidades_Diversas'].fillna(0) * 0.166 + 
                                    df['Puntaje_Semillas_Casa_Permanencia'].fillna(0) * 0.166 + 
                                    df['Puntaje_SIPEP'].fillna(0) * 0.166 +
                                    df['puntaje_diligenciamiento'].fillna(0) * 0.166
                                    )


In [ ]:
# Necesidad

df["Puntaje_RIESGO_TERRITORIAL"] = (df['Puntaje_Accesos y Condiciones Riesgos Naturales'].fillna(0) * 0.50 + 
                                    df['Puntaje_Accesos y Condiciones Riesgos Antrópicos'].fillna(0) * 0.50)


In [ ]:
# Cumplimiento 0.30 y Necesidad 0.175 x 4

df["Puntaje_RIESGO_ESTRUCTURAL"] = (df['Puntaje_mantenimiento_infraestructura'].fillna(0) * 0.30 +
                                    df['Puntaje_antiguedad'].fillna(0) * 0.175 +
                                    df['Puntaje_antiguedad_espacio_abierto'].fillna(0) * 0.175 +
                                    df['Puntaje_estado_actual'].fillna(0) * 0.175 + 
                                    df['Puntaje_estado_actual_espacio_abierto'].fillna(0) * 0.175
                                    )

In [ ]:
# Necesidad

df["Puntaje_DEFICIT_COBERTURA"] = (df['Puntaje_ocupacion_uso'].fillna(0) * 0.333 +
                                    df['Puntaje_uso_espacio_abierto'].fillna(0) * 0.333 +
                                    df['Puntaje_hacinamiento'].fillna(0) * 0.333 
                                    )

In [ ]:
# Cumplimiento

df["Puntaje_DEFICIT_CUALITATIVO"] = (df['Puntaje_arquitectura_propia'].fillna(0) * 0.20 + 
                                        df['Puntaje_tipo_arquitectura'].fillna(0) * 0.20 +
                                        df['Puntaje_planimetria_sede'].fillna(0) * 0.20 + 
                                        df['Puntaje_muros'].fillna(0) * 0.20 +
                                        df['Puntaje_fachada'].fillna(0) * 0.20
                                        )

In [ ]:
# Necesidad

df["Puntaje_INSTALACIONES_TECNICAS"] = (df['Puntaje_Condiciones_Servicio_Agua'].fillna(0) * 0.055 +
                                        df['Puntaje_componentes_presentes_suministro_agua'].fillna(0) * 0.055 +
                                        df['Puntaje_componentes_presentes_alcantarillado'].fillna(0) * 0.055 +
                                        df['Puntaje_tipo_aparato_hidrosanitario'].fillna(0) * 0.055 +
                                        df['Puntaje_aguas_lluvias'].fillna(0) * 0.055 +
                                        df['Puntaje_suministro_gas'].fillna(0) * 0.055 +
                                        df['Puntaje_suministro_energia'].fillna(0) * 0.055 +
                                        df['Puntaje_estado_energia_solar'].fillna(0) * 0.055 +
                                        df['Puntaje_estado_planta_electrica'].fillna(0) * 0.055 +
                                        df['Puntaje_acometida_principal'].fillna(0) * 0.055 +
                                        df['Puntaje_estado_acometida_electrica'].fillna(0) * 0.055 +
                                        df['Puntaje_estado_iluminacion'].fillna(0) * 0.055 +
                                        df['Puntaje_estado_tomacorrientes'].fillna(0) * 0.055 +
                                        df['Puntaje_estado_tablero'].fillna(0) * 0.055 +
                                        df['Puntaje_estado_circuitos'].fillna(0) * 0.055 +
                                        df['Puntaje_material_tuberia_electrica'].fillna(0) * 0.055 +
                                        df['Puntaje_tipo_comunicacion'].fillna(0) * 0.055 +
                                        df['Puntaje_estado_comunicaciones'].fillna(0) * 0.055 
                                        )

In [ ]:
#Cumplimiento 0.06 y Necesidad 0.14 x 5

df["Puntaje_ACCESIBILIDAD"] = (df['Puntaje_Estado_Predio'].fillna(0) * 0.06 +
                                df['Puntaje_Predio_Arrendado'].fillna(0) * 0.06 +
                                df['Puntaje_Medios_Transporte'].fillna(0) * 0.06 +
                                df['Puntaje_Accesos_Condiciones_Predio'].fillna(0) * 0.14 +
                                df['Puntaje_Accesos y Condiciones Cerramiento Perimetral'].fillna(0) * 0.14 +
                                df['Puntaje_acceso_semillas_diversas'].fillna(0) * 0.14 +
                                df['Puntaje_elementos_complementarios'].fillna(0) * 0.14 +
                                df['Puntaje_condicion_fisica_ruta'].fillna(0) * 0.14 +
                                df['Puntaje_plan_evacuacion'].fillna(0) * 0.06 +
                                df['Puntaje_rutas_evacuacion'].fillna(0) * 0.06 +
                                df['Puntaje_simulacros_evacuacion'].fillna(0) * 0.06 
                                )

In [ ]:
# Puntaje Cumplimiento 0.1666

df["Puntaje_CONFORT_AMBIENTAL"] = (df['Puntaje_huerta_escolar'].fillna(0) * 0.1666 +
                                    df['Puntaje_granja_escolar'].fillna(0) * 0.1666 +
                                    df['Puntaje_procesos_madre_tierra'].fillna(0) * 0.1666 +
                                    df['Puntaje_residuos_inorganicos'].fillna(0) * 0.1666 +
                                    df['Puntaje_residuos_organicos'].fillna(0) * 0.1666 +
                                    df['Puntaje_iluminaicion_exterior'].fillna(0) * 0.1666 
                                    )

In [ ]:
# peso porcentaje por modulo 12.5 %

df["Puntaje_TOTAL"]  = (df["Puntaje_CULTURAL_PEDAGOGICO"].fillna(0) * 0.125 +
                        df['Puntaje_RIESGO_TERRITORIAL'].fillna(0) * 0.125 +
                        df['Puntaje_RIESGO_ESTRUCTURAL'].fillna(0) * 0.125 +
                        df['Puntaje_DEFICIT_COBERTURA'].fillna(0) * 0.125 +
                        df['Puntaje_DEFICIT_CUALITATIVO'].fillna(0) * 0.125 +
                        df['Puntaje_INSTALACIONES_TECNICAS'].fillna(0) * 0.125 +
                        df['Puntaje_ACCESIBILIDAD'].fillna(0) * 0.125 +
                        df['Puntaje_CONFORT_AMBIENTAL'].fillna(0) * 0.125
                        )

In [ ]:
n = len(df['Nombre'])

# Aumentar separación entre sedes
x = np.arange(n) * 3

# Barras más delgadas
width = 0.12

plt.figure(figsize=(18,6))

plt.bar(x - 4*width, df['Puntaje_CULTURAL_PEDAGOGICO'], width, label='Cultural Pedagógico')
plt.bar(x - 3*width, df['Puntaje_RIESGO_TERRITORIAL'], width, label='Riesgo Territorial')
plt.bar(x - 2*width, df['Puntaje_RIESGO_ESTRUCTURAL'], width, label='Riesgo Estructural')
plt.bar(x - width, df['Puntaje_DEFICIT_COBERTURA'], width, label='Déficit Cobertura')
plt.bar(x, df['Puntaje_DEFICIT_CUALITATIVO'], width, label='Déficit Cualitativo')
plt.bar(x + width, df['Puntaje_INSTALACIONES_TECNICAS'], width, label='Instalaciones Técnicas')
plt.bar(x + 2*width, df['Puntaje_ACCESIBILIDAD'], width, label='Accesibilidad')
plt.bar(x + 3*width, df['Puntaje_CONFORT_AMBIENTAL'], width, label='Confort Ambiental')
plt.bar(x + 4*width, df['Puntaje_TOTAL'], width, label='Total')

plt.title('PUNTAJE POR MODULOS', fontsize=14)
plt.xlabel('Sedes Educativas')
plt.ylabel('Valor Puntaje')

plt.xticks(x, df['Nombre'], rotation=90)
plt.ylim(0,100)

plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.axhline(
    y=49,
    linestyle='--',
    linewidth=2,
    color='red',
    label='Media de Priorización'
)

plt.legend(bbox_to_anchor=(1.02,1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# Menores de 30
sin_diligenciar = df[df['Puntaje_TOTAL'] < 30]

# Cercanas a 49 (por ejemplo ±5 puntos)
cerca_49 = df[
    (df['Puntaje_TOTAL'] >= 44) &
    (df['Puntaje_TOTAL'] <= 54)
]

# Entre 49 y 74
entre_49_74 = df[
    (df['Puntaje_TOTAL'] > 49) &
    (df['Puntaje_TOTAL'] < 74)
]

# Entre 74 y 100
entre_74_100 = df[
    (df['Puntaje_TOTAL'] >= 74) &
    (df['Puntaje_TOTAL'] <= 100)
]

print("Menor de 30:",len(sin_diligenciar))
print("Cerca de 49:",len(cerca_49))
print("Entre 49 y 74:",len(entre_49_74))
print("Entre 74 y 100:",len(entre_74_100))

In [ ]:
import matplotlib.pyplot as plt

categorias = [
    "Menor a 30",
    "Cerca a 49",
    "Entre 49–74",
    "Entre 74–100"
]

valores = [
    len(sin_diligenciar),
    len(cerca_49),
    len(entre_49_74),
    len(entre_74_100)
]

plt.figure(figsize=(10,6))

barras = plt.bar(categorias, valores)

plt.title("CLASIFICACIÓN DE SEDES POR PUNTAJE")
plt.ylabel("Cantidad de Sedes Educativas")

# etiquetas encima
for barra in barras:
    altura = barra.get_height()

    plt.text(
        barra.get_x()+barra.get_width()/2,
        altura,
        str(int(altura)),
        ha='center'
    )

plt.grid(axis='y',alpha=0.3)

plt.show()

In [ ]:
print("SEDES SIN DILIGENCIAMIENTO")
sin_diligenciar[['Nombre Sede','Puntaje_TOTAL']].sort_values('Puntaje_TOTAL').head(10)

In [ ]:
print("SEDES ENTRE 49 Y 74 PUNTOS")
entre_49_74[['Nombre Sede', 'Zona','Resguardo', 'Municipio', 'Puntaje_TOTAL']].sort_values('Puntaje_TOTAL').head(10)

In [ ]:
print('TOP 10 SEDES CERCA A 49 PUNTOS')
cerca_49[['Nombre','Zona', 'Resguardo', 'Municipio','Puntaje_TOTAL']].sort_values('Puntaje_TOTAL').head(10).sort_values(by='Puntaje_TOTAL')

In [ ]:
print('SEDES NORTE')
cerca_49_norte = cerca_49[cerca_49['Zona'] == 'ZONA NORTE']
cerca_49_norte[['Nombre Sede', 'Zona','Resguardo','Municipio','Puntaje_TOTAL']].head(4).sort_values(by='Puntaje_TOTAL')

In [ ]:
print('SEDES CENTRO')
cerca_49_centro = cerca_49[cerca_49['Zona'] == 'ZONA CENTRO']
cerca_49_centro[['Nombre Sede', 'Zona','Resguardo','Municipio','Puntaje_TOTAL']].head().sort_values(by='Puntaje_TOTAL')

In [ ]:
print('SEDES TIERRADENTRO')
cerca_49_tierradentro = cerca_49[cerca_49['Zona'] == 'TIERRA DENTRO']
cerca_49_tierradentro[['Nombre Sede', 'Zona','Resguardo','Municipio','Puntaje_TOTAL']].head().sort_values(by='Puntaje_TOTAL')

In [ ]:
print('SEDES PUEBLO YANACONA')
cerca_49_sur = cerca_49[(cerca_49['Zona'] == 'ZONA SUR') & (cerca_49['Pueblo'] == 'Pueblo Yanakuna')]
cerca_49_sur[['Nombre Sede', 'Zona','Resguardo','Municipio','Puntaje_TOTAL']].sort_values(by='Puntaje_TOTAL')

In [ ]:
print('SEDES EDUCATIVAS SATH TAMA KIWE ENTRE 49 Y 74 PUNTOS')
entre_49_74_sur_2 = entre_49_74[(entre_49_74['Zona'] == 'ZONA SUR') & (entre_49_74['Puntaje_TOTAL'] < 60)]
entre_49_74_sur_2[['Nombre Sede', 'Zona','Resguardo','Municipio','Puntaje_TOTAL']].head(2).sort_values(by='Puntaje_TOTAL')

In [ ]:
print('SEDES SATH TAMA KIWE CERCA A 49 PUNTOS')
cerca_49_stk = cerca_49[(cerca_49['Zona'] == 'ZONA SATH TAMA KIWE')]
cerca_49_stk[['Nombre Sede', 'Zona','Resguardo','Municipio','Puntaje_TOTAL']]

In [ ]:
print('SEDES EDUCATIVAS SATH TAMA KIWE ENTRE 49 Y 74 PUNTOS')
entre_49_74_stk = entre_49_74[(entre_49_74['Zona'] == 'ZONA SATH TAMA KIWE') & (entre_49_74['Puntaje_TOTAL'] < 60)]
entre_49_74_stk[['Nombre Sede', 'Zona','Resguardo','Municipio','Puntaje_TOTAL']].head(6).sort_values(by='Puntaje_TOTAL')

In [ ]:
print('SEDES ORIENTE COTAINDOC')
cerca_49_oriente = cerca_49[(cerca_49['Zona'] == 'ZONA ORIENTE')]
cerca_49_oriente[['Nombre Sede', 'Zona','Resguardo','Municipio','Puntaje_TOTAL']].sort_values(by='Puntaje_TOTAL')

In [ ]:
print('SEDES EDUCATIVAS PUEBLO TOTOROEZ 49 PUNTOS')
cerca_49_pt = cerca_49[(cerca_49['Zona'] == 'PUEBLO TOTOROEZ') ]
cerca_49_pt[['Nombre Sede', 'Zona','Resguardo','Municipio','Puntaje_TOTAL']].sort_values(by='Puntaje_TOTAL')

In [ ]:
print('SEDES EDUCATIVAS PUEBLO TOTOROEZ ENTRE 49 Y 74 PUNTOS')
entre_49_74_pt = entre_49_74[(entre_49_74['Zona'] == 'PUEBLO TOTOROEZ')]
entre_49_74_pt[['Nombre Sede', 'Zona','Resguardo','Municipio','Puntaje_TOTAL']].head(1).sort_values(by='Puntaje_TOTAL')

In [ ]:
cerca_49_uwv = cerca_49[(cerca_49['Zona'] == 'ZONA UH WALA VXIC')]
cerca_49_uwv[['Nombre Sede', 'Zona','Resguardo','Municipio','Puntaje_TOTAL']].sort_values(by='Puntaje_TOTAL')

In [ ]:

cerca_49_nu = cerca_49[(cerca_49['Zona'] == 'ZONA NASA UUS')]
cerca_49_nu[['Nombre Sede', 'Zona','Resguardo','Municipio','Puntaje_TOTAL']].head().sort_values(by='Puntaje_TOTAL')

In [ ]:

print('SEDES NASA USS')
entre_49_74_nu = entre_49_74[(entre_49_74['Zona'] == 'ZONA NASA UUS')]
entre_49_74_nu[['Nombre Sede', 'Zona','Resguardo','Municipio','Puntaje_TOTAL']].head(4).sort_values(by='Puntaje_TOTAL')

In [ ]:
puntaje = df['Puntaje_TOTAL']

print("Cantidad:", puntaje.count())
print("Promedio:", round(puntaje.mean(),2))
print("Mediana:", round(puntaje.median(),2))
print("Moda:", puntaje.mode().values)
print("Mínimo:", puntaje.min())
print("Máximo:", puntaje.max())

print("Desviación estándar:",
    round(puntaje.std(),2))

print("Varianza:",
    round(puntaje.var(),2))

print("Cuartil 25:",
    round(puntaje.quantile(0.25),2))

print("Cuartil 75:",
    round(puntaje.quantile(0.75),2))

print("P10:",puntaje.quantile(.10))
print("P25:",puntaje.quantile(.25))
print("P50:",puntaje.quantile(.50))
print("P75:",puntaje.quantile(.75))
print("P90:",puntaje.quantile(.90))

In [ ]:
import matplotlib.pyplot as plt

puntaje = df['Puntaje_TOTAL'].dropna()

# Calcular percentiles
p25 = puntaje.quantile(0.25)
p50 = puntaje.quantile(0.50)
p75 = puntaje.quantile(0.75)

# Cantidad de sedes por rango
q1 = len(puntaje[puntaje <= p25])

q2 = len(
    puntaje[
        (puntaje > p25) &
        (puntaje <= p50)
    ]
)

q3 = len(
    puntaje[
        (puntaje > p50) &
        (puntaje <= p75)
    ]
)

q4 = len(puntaje[puntaje > p75])

plt.figure(figsize=(14,3))

plt.boxplot(
    puntaje,
    vert=False,
    showmeans=True
)

# líneas de percentiles
plt.axvline(p25, linestyle='--', label='P25')
plt.axvline(p50, linestyle='--', label='P50 Mediana')
plt.axvline(p75, linestyle='--', label='P75')

# texto con cantidades
plt.text(p25/2,
        1.15,
        f"{q1} sedes")

plt.text((p25+p50)/2,
        1.15,
        f"{q2} sedes")

plt.text((p50+p75)/2,
        1.15,
        f"{q3} sedes")

plt.text((p75+100)/2,
        1.15,
        f"{q4} sedes")

plt.xlim(0,100)

plt.xlabel("Puntaje Total")
plt.title("Distribución de sedes por percentiles")

plt.legend()

plt.show()